In [4]:
import pandas as pd
import numpy as np
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Connection string
conn_str = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

# SQL query
query = '''
SELECT
[call_type],
[priority],
[initial_call_type_mapping],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
[call_sign_total_service_time_s]
from [gt].[dbo].[call_data_20251019_processed_v44]
tablesample (10 percent)
'''

# Fetch data
with pyodbc.connect(conn_str) as conn:
    df = pd.read_sql(query, conn)

# Convert date to Unix timestamp (seconds since epoch)
df['cad_event_original_time_queued_date'] = pd.to_datetime(df['cad_event_original_time_queued_date']).astype(np.int64) // 10**9

# Drop missing target values
df = df.dropna(subset=['call_sign_total_service_time_s'])

# Categorical columns
categorical_cols = [
    'call_type', 'priority', 'initial_call_type_mapping', 
    'dispatch_precinct', 'dispatch_sector', 'dispatch_beat',
    'dispatch_reporting_area', 'cad_event_response_category',
    'call_type_indicator', 'dispatch_neighborhood',
    'call_type_received_classification'
]

# One-hot encoding for categorical variables
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Split features and target
X = df.drop('call_sign_total_service_time_s', axis=1)
y = df['call_sign_total_service_time_s']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build neural network model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)  # Linear activation for regression
])

# Compile model
model.compile(optimizer='adam', loss='mse')

# Early stopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=10)

# Train model
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

# Evaluate model
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Neural Network RMSE: {rmse:.2f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_21592\3960960895.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\RQ\anaconda3\envs\env_python\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Neural Network RMSE: 2432.53
